# Análisis de Relaciones: Crímenes, Estaciones de Policía y Transporte en Nueva York

En este notebook analizamos las relaciones espaciales entre:
- **Crímenes** registrados en Nueva York
- **Estaciones de policía**
- **Estaciones de transporte**

Evaluaremos si existe correlación entre la ubicación de estas infraestructuras y la incidencia delictiva.

**Requisito:** Ejecutar primero `pipeline.ipynb` (Steps 1-2) para descargar y normalizar los datos.

In [1]:
!pip install -q pyspark==3.4.2 apache-sedona==1.6.1 folium

In [2]:
import os
from pyspark.sql.functions import col, expr, array, concat, lit
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
import folium

from src.config import (
    NY_CRIMES_NORMALIZED, NY_POLICE_STATIONS, NY_TRANSPORT_STATIONS,
    KMEANS_K, KMEANS_SEED, FP_MIN_SUPPORT, FP_MIN_CONFIDENCE, PROXIMITY_THRESHOLD_KM,
)
from src.spark_session import get_or_create_spark

spark = get_or_create_spark()

## 1. Carga de datos normalizados

In [3]:
# Cargar datos normalizados de crímenes
df_crimes = spark.read.parquet(NY_CRIMES_NORMALIZED)
print(f"Crímenes NY: {df_crimes.count():,} registros")
df_crimes.printSchema()
df_crimes.select("complaint_id", "offense_description", "latitude", "longitude", "crime_start_date").show(5, False)

Crímenes NY: 3,617 registros
root
 |-- complaint_id: string (nullable = true)
 |-- precinct_code: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- crime_start_date: date (nullable = true)
 |-- crime_start_time: string (nullable = true)
 |-- crime_end_date: date (nullable = true)
 |-- crime_end_time: string (nullable = true)
 |-- report_date: date (nullable = true)
 |-- crime_status: string (nullable = true)
 |-- jurisdiction_description: string (nullable = true)
 |-- offense_level: string (nullable = true)
 |-- offense_description: string (nullable = true)
 |-- internal_description: string (nullable = true)
 |-- premise_type: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- patrol_borough: string (nullable = true)
 |-- suspect_age_group: string (nullable = true)
 |-- suspect_race: string (nullable = true)
 |-- suspect_sex: string (nullable = true)
 |-- victim_age_group: string (nullable = true)
 |-- victim_race: string (nullable = true)
 |

In [4]:
# Cargar estaciones de policía
df_police = spark.read.option("header", True).csv(NY_POLICE_STATIONS)
print(f"Estaciones de policía: {df_police.count()} registros")

# Cargar estaciones de transporte
if os.path.exists(NY_TRANSPORT_STATIONS):
    df_transport = spark.read.option("header", True).csv(NY_TRANSPORT_STATIONS)
    print(f"Estaciones de transporte: {df_transport.count()} registros")
else:
    print(f"AVISO: No se encontró {NY_TRANSPORT_STATIONS}")
    print("Ejecutar pipeline.ipynb Step 1 para descargar las estaciones de transporte.")
    df_transport = None

Estaciones de policía: 1805 registros
Estaciones de transporte: 5870 registros


## 2. Preparación de datos geoespaciales

In [5]:
from sedona.register import SedonaRegistrator
SedonaRegistrator.registerAll(spark)

# Filtrar coordenadas válidas de Nueva York
df_crimes_clean = (
    df_crimes
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())
    .filter(col("latitude") != 0)
    .filter(col("longitude") != 0)
    .filter((col("latitude") > 40.4) & (col("latitude") < 41.0))
    .filter((col("longitude") > -74.3) & (col("longitude") < -73.7))
)

print(f"Crímenes con coordenadas válidas: {df_crimes_clean.count():,}")

# Crear geometrías
df_crimes_geo = df_crimes_clean.withColumn(
    "geometry",
    expr("ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE))")
)

/tmp/ipykernel_106/2525418085.py:2: DeprecationWarning: Call to deprecated function registerAll (Deprecated since 1.4.1, use SedonaContext.create() instead.).
  SedonaRegistrator.registerAll(spark)


Crímenes con coordenadas válidas: 3,617


In [6]:
# Filtrar estaciones en el área de Nueva York
df_police_ny = (
    df_police
    .filter((col("lat").cast("double") > 40.4) & (col("lat").cast("double") < 41.0))
    .filter((col("lon").cast("double") > -74.3) & (col("lon").cast("double") < -73.7))
)
df_police_geo = df_police_ny.withColumn(
    "geometry", expr("ST_Point(CAST(lon AS DOUBLE), CAST(lat AS DOUBLE))")
)
print(f"Estaciones de policía en NY: {df_police_geo.count()}")

if df_transport is not None:
    df_transport_ny = (
        df_transport
        .filter((col("lat").cast("double") > 40.4) & (col("lat").cast("double") < 41.0))
        .filter((col("lon").cast("double") > -74.3) & (col("lon").cast("double") < -73.7))
    )
    df_transport_geo = df_transport_ny.withColumn(
        "geometry", expr("ST_Point(CAST(lon AS DOUBLE), CAST(lat AS DOUBLE))")
    )
    print(f"Estaciones de transporte en NY: {df_transport_geo.count()}")
else:
    df_transport_geo = None

Estaciones de policía en NY: 214
Estaciones de transporte en NY: 2814


## 3. Análisis de distancias y relaciones espaciales

In [7]:
from src.analyze import compute_spatial_distances, categorize_proximity

if df_transport_geo is not None:
    df_distances = compute_spatial_distances(spark, df_crimes_geo, df_police_geo, df_transport_geo)
    df_distances.createOrReplaceTempView("crimes_with_distances")
    
    df_distances.select(
        "offense_description",
        "dist_to_nearest_police_km",
        "dist_to_nearest_transport_km"
    ).describe().show()

    df_distances.select(
        "offense_description",
        "dist_to_nearest_police_km",
        "dist_to_nearest_transport_km"
    ).show(10, False)
else:
    print("Saltando análisis de distancias: faltan estaciones de transporte.")

+-------+--------------------+-------------------------+----------------------------+
|summary| offense_description|dist_to_nearest_police_km|dist_to_nearest_transport_km|
+-------+--------------------+-------------------------+----------------------------+
|  count|                3617|                     3617|                        3617|
|   mean|                NULL|       1.7960104978837745|          1.2638845086178232|
| stddev|                NULL|       1.8640863708845157|          1.3768744591313806|
|    min| ADMINISTRATIVE CODE|     0.004660075941750078|        5.344867242663196E-4|
|    max|VEHICLE AND TRAFF...|        8.737885525393377|           6.204047724952155|
+-------+--------------------+-------------------------+----------------------------+

+------------------------------+-------------------------+----------------------------+
|offense_description           |dist_to_nearest_police_km|dist_to_nearest_transport_km|
+------------------------------+-----------------

## 4. Categorización de cercanía y reglas de asociación (FP-Growth)

In [8]:
if df_transport_geo is not None:
    df_categorized = categorize_proximity(df_distances)

    # Crear transacciones para FP-Growth
    transactions_ny = df_categorized.select(
        array(
            concat(lit("TIPO="), col("offense_description")),
            col("near_police"),
            col("near_transport")
        ).alias("items")
    )

    transactions_ny.show(10, False)

+----------------------------------------------------------------------+
|items                                                                 |
+----------------------------------------------------------------------+
|[TIPO=PETIT LARCENY, LEJOS_POLICIA, LEJOS_TRANSPORTE]                 |
|[TIPO=GRAND LARCENY OF MOTOR VEHICLE, LEJOS_POLICIA, LEJOS_TRANSPORTE]|
|[TIPO=RAPE, CERCA_POLICIA, LEJOS_TRANSPORTE]                          |
|[TIPO=PETIT LARCENY, LEJOS_POLICIA, LEJOS_TRANSPORTE]                 |
|[TIPO=CRIMINAL MISCHIEF & RELATED OF, LEJOS_POLICIA, LEJOS_TRANSPORTE]|
|[TIPO=MISCELLANEOUS PENAL LAW, LEJOS_POLICIA, LEJOS_TRANSPORTE]       |
|[TIPO=GRAND LARCENY, LEJOS_POLICIA, CERCA_TRANSPORTE]                 |
|[TIPO=RAPE, CERCA_POLICIA, CERCA_TRANSPORTE]                          |
|[TIPO=GRAND LARCENY, CERCA_POLICIA, CERCA_TRANSPORTE]                 |
|[TIPO=THEFT-FRAUD, LEJOS_POLICIA, LEJOS_TRANSPORTE]                   |
+--------------------------------------------------

In [9]:
if df_transport_geo is not None:
    from src.analyze import run_fp_growth

    freq_itemsets, rules = run_fp_growth(df_categorized)

    print("=== PATRONES FRECUENTES ===")
    freq_itemsets.orderBy(col("freq").desc()).show(20, False)

    print("\n=== REGLAS DE ASOCIACIÓN ===")
    rules.orderBy(col("confidence").desc()).show(20, False)

=== PATRONES FRECUENTES ===
+-----------------------------------------------------+----+
|items                                                |freq|
+-----------------------------------------------------+----+
|[LEJOS_POLICIA]                                      |2748|
|[LEJOS_TRANSPORTE]                                   |2083|
|[LEJOS_TRANSPORTE, LEJOS_POLICIA]                    |1855|
|[CERCA_TRANSPORTE]                                   |1534|
|[CERCA_TRANSPORTE, LEJOS_POLICIA]                    |893 |
|[CERCA_POLICIA]                                      |869 |
|[TIPO=GRAND LARCENY]                                 |704 |
|[CERCA_POLICIA, CERCA_TRANSPORTE]                    |641 |
|[TIPO=GRAND LARCENY, LEJOS_POLICIA]                  |579 |
|[TIPO=PETIT LARCENY]                                 |557 |
|[TIPO=HARRASSMENT 2]                                 |496 |
|[TIPO=PETIT LARCENY, LEJOS_POLICIA]                  |460 |
|[TIPO=HARRASSMENT 2, LEJOS_POLICIA]                  |44

## 5. Análisis estadístico por tipo de crimen

In [10]:
if df_transport_geo is not None:
    spark.sql("""
    SELECT 
        offense_description as tipo_crimen,
        COUNT(*) as total_crimenes,
        ROUND(AVG(dist_to_nearest_police_km), 2) as dist_media_policia_km,
        ROUND(AVG(dist_to_nearest_transport_km), 2) as dist_media_transporte_km,
        SUM(CASE WHEN dist_to_nearest_police_km < 0.5 THEN 1 ELSE 0 END) as cerca_policia,
        SUM(CASE WHEN dist_to_nearest_transport_km < 0.5 THEN 1 ELSE 0 END) as cerca_transporte
    FROM crimes_with_distances
    GROUP BY offense_description
    ORDER BY total_crimenes DESC
    """).show(20, False)

+------------------------------+--------------+---------------------+------------------------+-------------+----------------+
|tipo_crimen                   |total_crimenes|dist_media_policia_km|dist_media_transporte_km|cerca_policia|cerca_transporte|
+------------------------------+--------------+---------------------+------------------------+-------------+----------------+
|GRAND LARCENY                 |704           |1.61                 |0.95                    |125          |358             |
|PETIT LARCENY                 |557           |1.92                 |1.32                    |97           |210             |
|HARRASSMENT 2                 |496           |2.33                 |1.7                     |47           |147             |
|SEX CRIMES                    |285           |0.07                 |0.63                    |285          |190             |
|ASSAULT 3 & RELATED OFFENSES  |227           |2.42                 |1.54                    |16           |80        

## 6. Clustering espacial de crímenes (K-Means)

In [11]:
from src.analyze import run_kmeans

model_kmeans, predictions_ny = run_kmeans(df_crimes_clean)

print("=== CENTROS DE CLUSTERS (ZONAS DE ALTA CRIMINALIDAD) ===")
for i, center in enumerate(model_kmeans.clusterCenters()):
    print(f"Cluster {i}: lon={center[0]:.4f}, lat={center[1]:.4f}")

print(f"\nCrímenes por cluster:")
predictions_ny.groupBy("prediction").count().orderBy("prediction").show()

=== CENTROS DE CLUSTERS (ZONAS DE ALTA CRIMINALIDAD) ===
Cluster 0: lon=-73.9791, lat=40.7556
Cluster 1: lon=-74.1900, lat=40.5461
Cluster 2: lon=-73.7701, lat=40.6878
Cluster 3: lon=-73.9999, lat=40.6204
Cluster 4: lon=-73.9184, lat=40.6701
Cluster 5: lon=-74.1268, lat=40.6123
Cluster 6: lon=-73.9996, lat=40.7122
Cluster 7: lon=-73.8691, lat=40.8563
Cluster 8: lon=-73.9269, lat=40.8155
Cluster 9: lon=-73.8532, lat=40.7353

Crímenes por cluster:
+----------+-----+
|prediction|count|
+----------+-----+
|         0|  289|
|         1|  440|
|         2|  134|
|         3|  355|
|         4|  273|
|         5| 1061|
|         6|  432|
|         7|  204|
|         8|  244|
|         9|  185|
+----------+-----+



## 7. Visualización en mapa interactivo

In [12]:
# Tomar muestras para visualización
sample_crimes = df_crimes_geo.select(
    col("latitude").cast("double"),
    col("longitude").cast("double"),
    col("offense_description")
).sample(fraction=0.01, seed=42)

sample_police = df_police_geo.select(
    col("lat").cast("double").alias("latitude"),
    col("lon").cast("double").alias("longitude"),
    col("name")
).toPandas()

pdf_crimes = sample_crimes.toPandas()

if df_transport_geo is not None:
    sample_transport = df_transport_geo.select(
        col("lat").cast("double").alias("latitude"),
        col("lon").cast("double").alias("longitude"),
        col("name"),
        col("transport_type")
    ).toPandas()
else:
    sample_transport = None

print(f"Muestra de crímenes para mapa: {len(pdf_crimes)}")
print(f"Estaciones de policía: {len(sample_police)}")
if sample_transport is not None:
    print(f"Estaciones de transporte: {len(sample_transport)}")

Muestra de crímenes para mapa: 30
Estaciones de policía: 214
Estaciones de transporte: 2814


In [13]:
# Crear mapa centrado en Nueva York
m_ny = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Agregar centros de clusters de criminalidad
for i, center in enumerate(model_kmeans.clusterCenters()):
    folium.Circle(
        location=[center[1], center[0]],
        radius=1000,
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.2,
        popup=f'Cluster criminalidad {i}',
        tooltip=f'Zona alta criminalidad {i}'
    ).add_to(m_ny)

# Agregar muestra de crímenes
for _, row in pdf_crimes.iterrows():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=2,
        color='darkred',
        fill=True,
        fillColor='red',
        fillOpacity=0.4,
        popup=row.offense_description,
        tooltip='Crimen'
    ).add_to(m_ny)

# Agregar estaciones de policía
for _, row in sample_police.iterrows():
    folium.Marker(
        location=[row.latitude, row.longitude],
        icon=folium.Icon(color='blue', icon='shield', prefix='fa'),
        popup=f"Policía: {row['name']}",
        tooltip='Estación de Policía'
    ).add_to(m_ny)

# Agregar estaciones de transporte
if sample_transport is not None:
    for _, row in sample_transport.iterrows():
        folium.Marker(
            location=[row.latitude, row.longitude],
            icon=folium.Icon(color='green', icon='train', prefix='fa'),
            popup=f"Transporte: {row['name']}<br>Tipo: {row['transport_type']}",
            tooltip='Estación de Transporte'
        ).add_to(m_ny)

# Leyenda
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 250px; height: 180px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><strong>Leyenda</strong></p>
<p><i class="fa fa-circle" style="color:red"></i> Crímenes (muestra 1%)</p>
<p><i class="fa fa-circle" style="color:red;opacity:0.2"></i> Zonas alta criminalidad</p>
<p><i class="fa fa-shield" style="color:blue"></i> Estaciones de Policía</p>
<p><i class="fa fa-train" style="color:green"></i> Estaciones de Transporte</p>
</div>
'''
m_ny.get_root().html.add_child(folium.Element(legend_html))

m_ny

## 8. Análisis de correlación y conclusiones

In [14]:
if df_transport_geo is not None:
    # Análisis de proximidad: contar crímenes por categoría de distancia
    spark.sql("""
    SELECT 
        CASE 
            WHEN dist_to_nearest_police_km < 0.25 THEN 'Muy cerca (<250m)'
            WHEN dist_to_nearest_police_km < 0.5 THEN 'Cerca (250-500m)'
            WHEN dist_to_nearest_police_km < 1.0 THEN 'Distancia media (500m-1km)'
            ELSE 'Lejos (>1km)'
        END as proximidad_policia,
        CASE 
            WHEN dist_to_nearest_transport_km < 0.25 THEN 'Muy cerca (<250m)'
            WHEN dist_to_nearest_transport_km < 0.5 THEN 'Cerca (250-500m)'
            WHEN dist_to_nearest_transport_km < 1.0 THEN 'Distancia media (500m-1km)'
            ELSE 'Lejos (>1km)'
        END as proximidad_transporte,
        COUNT(*) as num_crimenes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
    FROM crimes_with_distances
    GROUP BY 1, 2
    ORDER BY num_crimenes DESC
    """).show(20, False)

+--------------------------+--------------------------+------------+----------+
|proximidad_policia        |proximidad_transporte     |num_crimenes|porcentaje|
+--------------------------+--------------------------+------------+----------+
|Lejos (>1km)              |Lejos (>1km)              |1165        |32.21     |
|Lejos (>1km)              |Distancia media (500m-1km)|415         |11.47     |
|Distancia media (500m-1km)|Muy cerca (<250m)         |242         |6.69      |
|Lejos (>1km)              |Cerca (250-500m)          |239         |6.61      |
|Distancia media (500m-1km)|Cerca (250-500m)          |222         |6.14      |
|Muy cerca (<250m)         |Muy cerca (<250m)         |217         |6.00      |
|Muy cerca (<250m)         |Cerca (250-500m)          |199         |5.50      |
|Lejos (>1km)              |Muy cerca (<250m)         |190         |5.25      |
|Distancia media (500m-1km)|Distancia media (500m-1km)|158         |4.37      |
|Cerca (250-500m)          |Muy cerca (<

In [15]:
if df_transport_geo is not None:
    # Resumen estadístico final
    print("=" * 70)
    print("RESUMEN DEL ANÁLISIS DE RELACIONES")
    print("=" * 70)

    cerca_policia = df_categorized.filter(col("near_police") == "CERCA_POLICIA").count()
    cerca_transporte = df_categorized.filter(col("near_transport") == "CERCA_TRANSPORTE").count()
    total = df_categorized.count()

    print(f"\nTotal de crímenes analizados: {total:,}")
    print(f"\nCrímenes cerca de estaciones de policía (<500m): {cerca_policia} ({cerca_policia*100/total:.2f}%)")
    print(f"Crímenes cerca de estaciones de transporte (<500m): {cerca_transporte} ({cerca_transporte*100/total:.2f}%)")

    cerca_ambas = df_categorized.filter(
        (col("near_police") == "CERCA_POLICIA") & 
        (col("near_transport") == "CERCA_TRANSPORTE")
    ).count()

    print(f"\nCrímenes cerca de AMBAS infraestructuras: {cerca_ambas} ({cerca_ambas*100/total:.2f}%)")
    print("\n" + "=" * 70)

RESUMEN DEL ANÁLISIS DE RELACIONES

Total de crímenes analizados: 3,617

Crímenes cerca de estaciones de policía (<500m): 869 (24.03%)
Crímenes cerca de estaciones de transporte (<500m): 1534 (42.41%)

Crímenes cerca de AMBAS infraestructuras: 641 (17.72%)

